# Data Enhancing – Magnetism (JARVIS-DFT)

Raw database → one labeled row per unique composition → 142 composition descriptors (Magpie + stoichiometry + valence orbitals).

* **Source:** JARVIS-DFT 3D (NIST)
* **Target:** is_magnetic (|magmom| >= 0.5 muB, ground-state polymorph)

In [1]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")
TASK = "magnetism"
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"

In [2]:
# Cell 1: Load the raw database
raw = pd.read_parquet(RAW / "jarvis/jarvis_dft_3d.parquet")
print(f"raw entries: {len(raw):,}")
raw.head()

raw entries: 93,902


,jid,formula,formation_energy_peratom,ehull,optb88vdw_bandgap,mbj_bandgap,magmom_oszicar,magmom_outcar,spg_number,crys
0,JVASP-90856,TiCuSiAs,-0.42762,2.1034,0.000,NaN,0.0,0.0,129,tetragonal
1,JVASP-86097,DyB6,-0.41596,4.5485,0.000,NaN,0.0,0.0,221,cubic
2,JVASP-64906,Be2OsRu,0.04847,3.5429,0.000,NaN,0.0,0.0,119,tetragonal
3,JVASP-98225,KBi,-0.44140,0.1629,0.472,NaN,0.0,0.0,14,monoclinic
4,JVASP-10,VSe2,-0.71026,1.8752,0.000,0.0,0.0,0.0,164,trigonal


In [3]:
# Cell 2: Simple stats
raw.describe(include='all').T.head(15)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
jid,93902,93902,JVASP-90856,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
formula,93902,65395,Li4MnCo2O7,102,NaN,NaN,NaN,NaN,NaN,NaN,NaN
formation_energy_peratom,93902.0,NaN,NaN,NaN,-0.832895,1.047505,-4.42304,-1.50686,-0.55899,-0.154872,5.35967
ehull,93902.0,NaN,NaN,NaN,1.855743,1.424971,0.0,0.6555,1.7189,2.7339,9.2516
optb88vdw_bandgap,93902.0,NaN,NaN,NaN,0.578425,1.310606,0.0,0.0,0.0,0.243,18.179
mbj_bandgap,21559.0,NaN,NaN,NaN,1.388805,2.302345,0.0,0.0,0.0,2.2455,32.563
magmom_oszicar,88811.0,NaN,NaN,NaN,0.768942,1.934949,0.0,0.0,0.0,0.0,10.0
magmom_outcar,92170.0,NaN,NaN,NaN,1.254329,3.713015,-68.529,0.0,0.0,0.0,75.388
spg_number,93902,215,225,9383,NaN,NaN,NaN,NaN,NaN,NaN,NaN
crys,93902,7,cubic,22697,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Cell 3: One label per unique composition (reduced formula, polymorph rule)
import prepare_datasets as P
lab = P.labeled_magnetism()
print(f"unique compositions: {len(lab):,}")
print(lab["target"].value_counts().rename({0: "negative", 1: "positive"}))
lab.head()

unique compositions: 64,410
target
negative    51157
positive    13253
Name: count, dtype: int64


,formula,target,magmom
0,Ac,0,0.0
1,Ac(AgGe)2,0,0.0
2,Ac(AlAg)2,0,0.0
3,Ac(AlSi)2,0,0.0
4,Ac(BIr)2,0,0.0


In [5]:
# Cell 4: Composition featurisation (vectorised Magpie, identical to matminer)
from featurize import featurize_formulas, feature_names
examples = pd.Series(["LiFePO4", "Fe2O3", "GaAs", "NdFeB", "SrTiO3"])
X, ok = featurize_formulas(examples)
X.index = examples
print(len(feature_names()), "descriptors")
X[["MagpieData mean Electronegativity", "MagpieData range Electronegativity", "MagpieData mean CovalentRadius", "MagpieData mean NValence", "MagpieData mean GSmagmom", "0-norm", "frac d valence electrons"]]

142 descriptors


,MagpieData mean Electronegativity,MagpieData range Electronegativity,MagpieData mean CovalentRadius,MagpieData mean NValence,MagpieData mean GSmagmom,0-norm,frac d valence electrons
LiFePO4,2.680,2.46,90.142857,5.428571,0.301523,4.0,0.157895
Fe2O3,2.796,1.61,92.400000,6.800000,0.844265,2.0,0.352941
GaAs,1.995,0.37,120.500000,14.000000,0.000000,2.0,0.714286
NdFeB,1.670,0.90,139.000000,5.666667,0.703554,3.0,0.352941
SrTiO3,2.562,2.49,110.600000,4.800000,0.000005,3.0,0.083333


In [6]:
# Cell 5: Check against matminer's reference implementation
from matminer.featurizers.composition import ElementProperty, Stoichiometry, ValenceOrbital
from pymatgen.core import Composition
ep, st, vo = ElementProperty.from_preset("magpie"), Stoichiometry(), ValenceOrbital(props=["frac"])
ref = np.array([ep.featurize(Composition(f)) + st.featurize(Composition(f)) + vo.featurize(Composition(f)) for f in examples], float)
print("max |ours - matminer| =", np.nanmax(np.abs(X.values - ref)))

max |ours - matminer| = 2.2737367544323206e-13


In [7]:
# Cell 6: Featurise the whole labeled set and the GNoME unlabeled pool
import time
t = time.time()
feats, ok = featurize_formulas(lab['formula'])
print(f"featurised {ok.sum():,} compositions in {time.time()-t:.1f}s")
df = pd.read_parquet(PROC / "magnetism_features.parquet")
gn = pd.read_parquet(PROC / "unlabeled_gnome_features.parquet", columns=["formula"])
print(f"saved labeled table: {df.shape} | GNoME unlabeled pool: {len(gn):,}")

featurised 64,410 compositions in 2.9s


saved labeled table: (64410, 145) | GNoME unlabeled pool: 554,054
